# infer: GPU tests and benchmarks on a Kaggle T4

**Setup:** run `python scripts/make_kaggle_zip.py` locally, upload `dist/infer-server.zip` as a private
Kaggle dataset, and attach it here (Add Input -> Your Datasets). Accelerator **GPU T4 x2** (not P100:
vLLM needs compute capability 7.0+), Internet **on** (weights, pip packages and ShareGPT are downloaded).

`MODE` below picks what runs. The full benchmark is roughly 9 GPU hours, close to Kaggle's 12-hour session
limit, so it is split in two:

| MODE | What runs | GPU time |
|---|---|---|
| `probe` | GPU-only tests (fp16 near-ties, CUDA graphs), vLLM install, a few requests through every system | ~30 min |
| `part1` | GPU tests, systems A B C D, offline experiments (batch-1, static batching, KV, profiler), plots | ~5 h |
| `part2` | systems E (CUDA graphs) and F (vLLM), Tier 2 extras, vLLM cross-check, 30-min soak, plots | ~4.5 h |
| `part3` | chunked prefill before/after (mixed workload), demo GIF of 50 streams | ~40 min |
| `part4` | GPU tests, D at 8/32/128 users in flight, E (CUDA graphs) again with 128-token context buckets | ~1.5 h |
| `soak` | the 30-minute soak test alone | ~45 min |
| `part5` | GPU tests, E again with graph memory set aside from the KV pool | ~1 h |
| `part6` | E as finally configured: graphs up to batch 32, eager above | ~50 min |
| `full` | everything | ~10 h |

Every result file records its GPU minutes; `bench/results/raw/gpu_minutes.jsonl` keeps the running total.
Results end up in `/kaggle/working/results_<MODE>.zip` (notebook Output tab).

In [ ]:
MODE = "probe"     # probe | part1 | part2 | part3 | part4 | soak | part5 | part6 | full

PLAN = {
    "probe": dict(gpu_tests=True, vllm=True, smoke=True, systems="", experiments=False, soak=False, demo=False),
    "part1": dict(gpu_tests=True, vllm=False, smoke=False, systems="A B C D", experiments=True, soak=False, demo=False),
    "part2": dict(gpu_tests=False, vllm=True, smoke=False, systems="E F X", experiments=False, soak=True, demo=False),
    "part3": dict(gpu_tests=False, vllm=False, smoke=False, systems="K", experiments=False, soak=False, demo=True),
    "part4": dict(gpu_tests=True, vllm=False, smoke=False, systems="Dc E", experiments=False, soak=False, demo=False,
                  env="E_RATES=4,8,16,32,inf E_CONC=8,32,128"),
    "soak": dict(gpu_tests=False, vllm=False, smoke=False, systems="", experiments=False, soak=True, demo=False),
    "part5": dict(gpu_tests=True, vllm=False, smoke=False, systems="E", experiments=False, soak=False, demo=False,
                  env="E_RATES=4,8,16,32,inf E_CONC=8,32,128"),
    "part6": dict(gpu_tests=False, vllm=False, smoke=False, systems="E", experiments=False, soak=False, demo=False,
                  env="E_RATES=4,8,16,32,inf E_CONC=8,32,128"),
    "full": dict(gpu_tests=True, vllm=True, smoke=False, systems="A B C D E F X K", experiments=True, soak=True,
                 demo=True),
}[MODE]
print(MODE, PLAN)

import os, subprocess, sys, time, urllib.request


def sh(cmd: str) -> int:
    """Run a bash command, stream its output here, never raise: one failure must not end an unattended run."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc:
        print(f"[exit {rc}]", flush=True)
    return rc


def wait_http(url: str, proc=None, seconds: int = 600) -> bool:
    for _ in range(seconds // 2):
        try:
            urllib.request.urlopen(url, timeout=5)
            return True
        except Exception:
            if proc is not None and proc.poll() is not None:
                return False
            time.sleep(2)
    return False


T0 = time.time()

In [ ]:
sh("nvidia-smi; nvidia-smi --query-gpu=driver_version,name,memory.total --format=csv")
sh("""python -c "import torch, sys; print(sys.version); print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))" """)
sh("df -h /kaggle/working /tmp | cat; nproc; free -g | head -2")

## 1. Install from the uploaded dataset (pinned)
`/kaggle/input` is read-only and the benchmarks write results, so the project is copied to
`/kaggle/working/infer-server`. Kaggle's own CUDA build of PyTorch is kept; the rest comes from the lock file.

In [ ]:
import glob, shutil, zipfile

DEST = "/kaggle/working/infer-server"
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "src", "infer"))]
if found:
    src = found[0]
else:                                   # the zip was kept as-is: unpack it ourselves
    zips = glob.glob("/kaggle/input/**/infer-server*.zip", recursive=True)
    assert zips, "attach the infer-server dataset to this notebook (Add Input -> Your Datasets)"
    zipfile.ZipFile(zips[0]).extractall("/tmp/_unzipped")
    src = os.path.dirname(glob.glob("/tmp/_unzipped/**/pyproject.toml", recursive=True)[0])
print("project source:", src)
shutil.copytree(src, DEST, dirs_exist_ok=True)
os.chdir(DEST)
sh("cat SOURCE_DIGEST")
sh("pip install -q -r requirements.lock 2>&1 | grep -v -i -E 'warning|notice' | tail -5; pip install -q -e . --no-deps")
sh("""python -c "import infer, torch, transformers; print('infer ok; torch', torch.__version__, 'transformers', transformers.__version__)" """)

## 2. vLLM in its own virtual environment
Installing vLLM next to Kaggle's PyTorch causes version clashes, so it gets a venv in `/tmp` (outside the
saved output), built with `uv` because Kaggle's Python lacks `ensurepip`. Recent vLLM wheels may target a newer CUDA than Kaggle's driver supports, so the newest
release is tried first and older ones only if a CUDA matmul fails inside the venv. vLLM always runs with
`--dtype float16` (Turing has no bf16) and falls back to its Triton attention backend on a T4.

In [ ]:
VLLM = None
if PLAN["vllm"]:
    # Kaggle's Python has no ensurepip, so `python -m venv` cannot bootstrap pip: uv builds the venv
    # (with its own CPython 3.12, the version vLLM wheels are most tested on) and installs into it.
    sh("pip install -q uv 2>&1 | tail -1; uv --version")
    env_dir = "/tmp/vllm-env"
    for ver in ["", "0.11.0", "0.10.1.1", "0.9.2"]:          # "" = newest release
        spec = f"vllm=={ver}" if ver else "vllm"
        sh(f"rm -rf {env_dir} && uv venv -q --python 3.12 {env_dir}")
        if sh(f"uv pip install -q --no-cache --python {env_dir}/bin/python '{spec}' 2>&1 | tail -5; "
              f"test -x {env_dir}/bin/vllm"):
            continue
        check = ("import torch, vllm; x = torch.ones(64, 64, device='cuda', dtype=torch.float16); "
                 "print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.version.cuda, float((x @ x).float().sum()))")
        if sh(f'{env_dir}/bin/python -c "{check}"') == 0:
            VLLM = f"{env_dir}/bin/vllm"
            break
print("VLLM =", VLLM)

In [ ]:
if VLLM and (PLAN["smoke"] or PLAN["systems"]):
    # Prove vLLM serves Qwen2.5-0.5B on this T4 before anything depends on it.
    log = open("/kaggle/working/vllm_smoke.log", "w")
    p = subprocess.Popen([VLLM, "serve", "Qwen/Qwen2.5-0.5B-Instruct", "--dtype", "float16", "--max-model-len", "2048",
                          "--port", "8001"], stdout=log, stderr=subprocess.STDOUT)
    if wait_http("http://localhost:8001/health", p, 900):
        sh(f"{VLLM} bench serve --backend openai --base-url http://localhost:8001 --model Qwen/Qwen2.5-0.5B-Instruct "
           "--dataset-name random --random-input-len 512 --random-output-len 128 --ignore-eos --num-prompts 50 "
           "--request-rate 4 2>&1 | tail -30")
    else:
        print("vLLM server did not come up; tail of its log:")
        sh("tail -40 /kaggle/working/vllm_smoke.log")
        VLLM = None
    p.terminate()
    p.wait()
print("VLLM =", VLLM)

## 3. Correctness on the GPU
Measure the fp16 tolerance on this GPU (HF fp16, batch 1 vs batch 8), then: fp16 logits within tolerance,
golden divergences only at near-ties, batch invariance, CUDA-graph replay equal to eager in fp32.

In [ ]:
if PLAN["gpu_tests"]:
    sh("python baselines/make_golden.py --fp16-only --device cuda 2>&1 | grep -v -i warn | tail -3")
    sh("head -5 tests/fp16_tolerance.json")
    sh("python -m pytest -q -m gpu tests/test_gpu.py -p no:cacheprovider -rA 2>&1 | tail -40")

## 4. Probe only: a few requests through every system, plus the batch-1 and profiler scripts

In [ ]:
if PLAN["smoke"]:
    PROBE = "bench/results/probe"
    os.makedirs(f"{PROBE}/raw", exist_ok=True)
    M = "Qwen/Qwen2.5-0.5B-Instruct"
    ours = "python -m infer.serve --port 8000 --dtype float16 --max-model-len 2048 --no-request-log"
    servers = {"A": "python -m baselines.naive_hf --port 8000 --dtype float16",
               "B": ours + " --engine static --static-batch-size 16",
               "C": ours + " --cache-type contiguous",
               "D": ours,
               "E": ours + " --cuda-graphs"}
    if VLLM:
        servers["F"] = f"{VLLM} serve {M} --dtype float16 --max-model-len 2048 --gpu-memory-utilization 0.9 --port 8000"
    for name, cmd in servers.items():
        log = open(f"{PROBE}/server_{name}.log", "w")
        p = subprocess.Popen("exec " + cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, executable="/bin/bash")
        if wait_http("http://localhost:8000/health", p, 900):
            n = 8 if name == "A" else 64
            sh(f"python -m bench.load_gen --system {name} --workload fixed --num-requests {n} --rates 4,inf --runs 1 "
               f"--warmup 4 --out-dir {PROBE}/raw 2>&1 | grep -v -i warn")
        else:
            print(f"!! server {name} did not start")
        p.terminate()
        try:
            p.wait(timeout=60)
        except Exception:
            p.kill()
        sh(f"sleep 5; tail -5 {PROBE}/server_{name}.log")
    sh("python -m bench.batch1 --reps 2 2>&1 | grep -v -i warn | tail -8")
    sh("python -m bench.profile_decode --batch 32 --steps 20 2>&1 | grep -v -i warn | head -25")
    sh("python -m bench.profile_decode --batch 32 --steps 20 --cuda-graphs --tag graphs 2>&1 | grep -v -i warn | head -25")
    sh(f"python -m bench.plot --results-dir {PROBE} 2>&1 | tail -5; cat {PROBE}/results.md")

## 5. Benchmarks: every selected system at its budgeted load points, three runs; experiments; plots

In [ ]:
if PLAN["systems"] or PLAN["experiments"]:
    env = f"ONLY='{PLAN['systems'] or 'none'}' SKIP_EXPERIMENTS={0 if PLAN['experiments'] else 1} {PLAN.get('env', '')}"
    if VLLM:
        env += f" VLLM={VLLM}"
    sh(f"{env} bash bench/run_all.sh 2>&1 | tee /kaggle/working/run_all_{MODE}.log | grep -v '^INFO'")

## 6. Soak test: 30 minutes of random arrivals, no leaks

In [ ]:
if PLAN["soak"]:
    log = open("/kaggle/working/soak_server.log", "w")
    srv = subprocess.Popen("exec python -m infer.serve --port 8000 --dtype float16 --no-request-log", shell=True,
                           stdout=log, stderr=subprocess.STDOUT, executable="/bin/bash")
    if wait_http("http://localhost:8000/health", srv, 900):
        sh("python -m bench.soak --minutes 30 --rate 4 2>&1 | grep -v -i warn | tail -40")
    srv.terminate()
    srv.wait()

## 6b. Demo: 50 concurrent streams, recorded and rendered to a GIF

In [ ]:
if PLAN["demo"]:
    log = open("/kaggle/working/demo_server.log", "w")
    srv = subprocess.Popen("exec python -m infer.serve --port 8000 --dtype float16 --no-request-log", shell=True,
                           stdout=log, stderr=subprocess.STDOUT, executable="/bin/bash")
    if wait_http("http://localhost:8000/health", srv, 900):
        sh("python -m bench.demo --streams 50 --record bench/results/demo_frames.jsonl --quiet 2>&1 | tail -2")
        sh("python -m bench.render_gif bench/results/demo_frames.jsonl bench/results/demo.gif")
    srv.terminate()
    srv.wait()

## 7. Results

In [ ]:
from IPython.display import Image, Markdown, display

res = "bench/results/results.md"
if os.path.exists(res) and PLAN["systems"]:
    display(Markdown(open(res).read()))
    for f in sorted(glob.glob("bench/results/figures/*.png")):
        print(f)
        display(Image(f))
print(f"GPU session time: {(time.time() - T0) / 3600:.2f} h")

Download `results_<MODE>.zip` from the Output tab and unzip it into the project folder on your machine;
`python -m bench.plot` rebuilds every figure and table from the raw JSON.

In [ ]:
sh(f"cd {DEST} && rm -f /kaggle/working/results_{MODE}.zip && zip -qr /kaggle/working/results_{MODE}.zip "
   f"bench/results tests/fp16_tolerance.json SOURCE_DIGEST; ls -lh /kaggle/working/results_{MODE}.zip")